In [17]:
import numpy as np
import pandas as pd
import os, sys, glob, argparse
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams["pdf.use14corefonts"] = True
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind
from mne.stats import permutation_cluster_1samp_test
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm
import analysis_helpers as helper
from plotting_functions import make_barplot_points_precomputed, determine_symbol
from config import *
from scipy import stats

In [19]:
#trialwise_results = pd.read_csv(BEHAV_TRIALSERIES, index_col=0)
run_results       = pd.read_csv('results/final_results/behavioral_change_session.csv', index_col=0)
lm_df = run_results.copy()
lm_df['wmp_first'] = lm_df['subject_id'].isin(WM_FIRST).astype(int)

In [21]:
# --- Model: delta_BC ~ session_type * wmp_first (interaction) ---
# Remove IM session since it always comes first
lm_df = lm_df[lm_df['session_type'] != 'IM']
formula = 'delta_BC ~ session_type * wmp_first'
m3 = smf.ols(formula, data=lm_df).fit()
f_test3 = anova_lm(m3, typ=1)

In [22]:
m3.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:               delta_BC   R-squared:                       0.262
Model:                            OLS   Adj. R-squared:                  0.193
Method:                 Least Squares   F-statistic:                     3.786
Date:                Wed, 01 Apr 2026   Prob (F-statistic):             0.0197
Time:                        09:52:07   Log-Likelihood:                 14.490
No. Observations:                  36   AIC:                            -20.98
Df Residuals:                      32   BIC:                            -14.65
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
=================================================================================================
                                    coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------
Intercept                        -0.0050      0.061     -0.082      0.935      -0.129       0.119
session_type[T.WMP]               0.1050      0.086      1.224      0.230      -0.070       0.280
wmp_first                        -0.0030      0.081     -0.037      0.971      -0.169       0.163
session_type[T.WMP]:wmp_first     0.1230      0.115      1.068      0.293      -0.111       0.357
==============================================================================
Omnibus:                        1.375   Durbin-Watson:                   2.540
Prob(Omnibus):                  0.503   Jarque-Bera (JB):                1.322
Skew:                           0.369   Prob(JB):                        0.516
Kurtosis:                       2.421   Cond. No.                         7.25
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [24]:
def add_partial_eta_squared(anova_table):
        ss_res = anova_table.loc["Residual", "sum_sq"]
        anova_table["eta_p2"] = anova_table["sum_sq"] / (anova_table["sum_sq"] + ss_res)
        return anova_table
f_test3 = add_partial_eta_squared(f_test3)
f_test3

,df,sum_sq,mean_sq,F,PR(>F),eta_p2
session_type,1.0,0.27040,0.270400,9.182054,0.004809,0.222962
wmp_first,1.0,0.03042,0.030420,1.032981,0.317083,0.031271
session_type:wmp_first,1.0,0.03362,0.033620,1.141644,0.293299,0.034447
Residual,32.0,0.94236,0.029449,NaN,NaN,0.500000


In [3]:
main_results = pd.read_csv('results/results_public/main_results.csv', index_col=0)

In [4]:
main_results.head()

,subject_id,session_number,session_type,score,metric,n_voxels,wmp_first
0,avatarRT_sub_05,2,IM,0.035580,delta_EVR,1453.0,True
1,avatarRT_sub_05,2,IM,0.400000,delta_BC,1453.0,True
2,avatarRT_sub_05,3,WMP,0.050398,delta_EVR,1453.0,True
3,avatarRT_sub_05,3,WMP,0.560000,delta_BC,1453.0,True
4,avatarRT_sub_05,4,OMP,0.004348,delta_EVR,1453.0,True


In [5]:
wide = main_results.pivot_table(index=['subject_id','session_type','wmp_first'], columns='metric', values='score').reset_index()

formula = '0 + delta_BC ~ delta_EVR * C(session_type) + wmp_first'
model1 = smf.ols(formula, 
                     data=wide).fit(cov_type='cluster', 
                                  cov_kwds={'groups': wide['subject_id']})
formula2 = '0 + delta_BC ~ delta_EVR + C(session_type) + wmp_first'
model2 = smf.ols(formula2, 
                     data=wide).fit(cov_type='cluster', 
                                  cov_kwds={'groups': wide['subject_id']})

In [6]:
model1.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:               delta_BC   R-squared:                       0.738
Model:                            OLS   Adj. R-squared:                  0.704
Method:                 Least Squares   F-statistic:                     35.88
Date:                Tue, 31 Mar 2026   Prob (F-statistic):           9.71e-09
Time:                        16:12:58   Log-Likelihood:                 33.865
No. Observations:                  54   AIC:                            -53.73
Df Residuals:                      47   BIC:                            -39.81
Df Model:                           6                                         
Covariance Type:              cluster                                         
====================================================================================================
                                       coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------------
Intercept                            0.4769      0.027     17.543      0.000       0.424       0.530
C(session_type)[T.OMP]              -0.5225      0.042    -12.506      0.000      -0.604      -0.441
C(session_type)[T.WMP]              -0.3919      0.049     -7.968      0.000      -0.488      -0.296
wmp_first[T.True]                    0.0679      0.027      2.540      0.011       0.016       0.120
delta_EVR                           -2.0033      1.037     -1.932      0.053      -4.036       0.029
delta_EVR:C(session_type)[T.OMP]     1.2414      2.171      0.572      0.567      -3.013       5.496
delta_EVR:C(session_type)[T.WMP]     7.8508      2.790      2.814      0.005       2.383      13.319
==============================================================================
Omnibus:                        0.138   Durbin-Watson:                   2.376
Prob(Omnibus):                  0.933   Jarque-Bera (JB):                0.080
Skew:                           0.083   Prob(JB):                        0.961
Kurtosis:                       2.909   Cond. No.                         280.
==============================================================================

Notes:
[1] Standard Errors are robust to cluster correlation (cluster)
"""

In [13]:
model = smf.ols('delta_BC ~ delta_EVR ', 
                     data=wide[wide['session_type']=='IM']).fit()
model.summary()

/Users/elb/miniconda3/envs/rtcloud_av1/lib/python3.7/site-packages/scipy/stats/stats.py:1542: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=18
  "anyway, n=%i" % int(n))


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:               delta_BC   R-squared:                       0.082
Model:                            OLS   Adj. R-squared:                  0.025
Method:                 Least Squares   F-statistic:                     1.428
Date:                Tue, 31 Mar 2026   Prob (F-statistic):              0.250
Time:                        16:16:51   Log-Likelihood:                 19.304
No. Observations:                  18   AIC:                            -34.61
Df Residuals:                      16   BIC:                            -32.83
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.5111      0.025     20.053      0.000       0.457       0.565
delta_EVR     -1.6710      1.398     -1.195      0.250      -4.635       1.293
==============================================================================
Omnibus:                        0.833   Durbin-Watson:                   2.298
Prob(Omnibus):                  0.659   Jarque-Bera (JB):                0.684
Skew:                           0.005   Prob(JB):                        0.711
Kurtosis:                       2.045   Cond. No.                         67.6
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [29]:
# compare two models with a likelihood ratio test
from statsmodels.stats.anova import anova_lm
anova_results = anova_lm(model1, model2)
print(anova_results)

   df_resid       ssr  df_diff   ss_diff         F  Pr(>F)
0      47.0  0.901987      0.0       NaN       NaN     NaN
1      49.0  1.070948     -2.0 -0.168962  3.865321     NaN


In [30]:
from scipy.stats import chi2

LR_stat = 2 * (model1.llf - model2.llf)
df_diff = 1  # Difference in number of parameters (random intercept variance)
p_value = 0.5  * chi2.sf(LR_stat, df=df_diff)  # df=1 for one random effect parameter, variance is on the boundary of parameter space (non-negative)
print(f"Comparing LME and OLS models; LR statistic: {LR_stat:.3f}, p = {p_value:.3f}")

Comparing LME and OLS models; LR statistic: 9.272, p = 0.001


In [32]:
RSS_reduced = model2.ssr
RSS_full = model1.ssr
df1 = 2  # number of interaction terms
df2 = model1.df_resid  # should be 47

F_stat = ((RSS_reduced - RSS_full) / df1) / (RSS_full / df2)
p_value = 1 - stats.f.cdf(F_stat, df1, df2)

print(f"F({df1}, {df2}) = {F_stat:.3f}, p = {p_value:.3f}")

F(2, 47.0) = 4.402, p = 0.018


In [27]:
import traceback

def save_manifold_components(subject_ids=SUB_IDS, n_components=None):
    for sub in subject_ids:
        model_dir = f'{DATA_PATH}/{sub}/model'
        bottleneck_path = f'{model_dir}/bottleneck.npy'
        try:
            bottleneck = np.load(bottleneck_path)
            components = helper.get_manifold_component(bottleneck, n_components=n_components)
            for i, comp in enumerate(components):
                out_path = f'{model_dir}/manifold_pc_{i+1:02d}.npy'
                np.save(out_path, comp)
                print(f'Saved {out_path}')
        except Exception as e:
            print(f'{sub}: failed ({e})')
            traceback.print_exc()

save_manifold_components()

Saved /Users/elb/Desktop/BCI/avatarRT_dryad/avatarRT_subject_data/avatarRT_sub_05/model/manifold_pc_01.npy
Saved /Users/elb/Desktop/BCI/avatarRT_dryad/avatarRT_subject_data/avatarRT_sub_05/model/manifold_pc_02.npy
Saved /Users/elb/Desktop/BCI/avatarRT_dryad/avatarRT_subject_data/avatarRT_sub_05/model/manifold_pc_03.npy
Saved /Users/elb/Desktop/BCI/avatarRT_dryad/avatarRT_subject_data/avatarRT_sub_05/model/manifold_pc_04.npy
Saved /Users/elb/Desktop/BCI/avatarRT_dryad/avatarRT_subject_data/avatarRT_sub_05/model/manifold_pc_05.npy
Saved /Users/elb/Desktop/BCI/avatarRT_dryad/avatarRT_subject_data/avatarRT_sub_05/model/manifold_pc_06.npy
Saved /Users/elb/Desktop/BCI/avatarRT_dryad/avatarRT_subject_data/avatarRT_sub_05/model/manifold_pc_07.npy
Saved /Users/elb/Desktop/BCI/avatarRT_dryad/avatarRT_subject_data/avatarRT_sub_05/model/manifold_pc_08.npy
Saved /Users/elb/Desktop/BCI/avatarRT_dryad/avatarRT_subject_data/avatarRT_sub_05/model/manifold_pc_09.npy
Saved /Users/elb/Desktop/BCI/avatarRT

Traceback (most recent call last):
  File "/var/folders/1f/s3_5gwyj4j3c18swys7lqz4w0000gn/T/ipykernel_10328/1631134046.py", line 8, in save_manifold_components
    bottleneck = np.load(bottleneck_path)
  File "/Users/elb/miniconda3/envs/rtcloud_av1/lib/python3.7/site-packages/numpy/lib/npyio.py", line 417, in load
    fid = stack.enter_context(open(os_fspath(file), "rb"))
FileNotFoundError: [Errno 2] No such file or directory: '/Users/elb/Desktop/BCI/avatarRT_dryad/avatarRT_subject_data/avatarRT_sub_09/model/bottleneck.npy'
Traceback (most recent call last):
  File "/var/folders/1f/s3_5gwyj4j3c18swys7lqz4w0000gn/T/ipykernel_10328/1631134046.py", line 8, in save_manifold_components
    bottleneck = np.load(bottleneck_path)
  File "/Users/elb/miniconda3/envs/rtcloud_av1/lib/python3.7/site-packages/numpy/lib/npyio.py", line 417, in load
    fid = stack.enter_context(open(os_fspath(file), "rb"))
FileNotFoundError: [Errno 2] No such file or directory: '/Users/elb/Desktop/BCI/avatarRT_dryad/